<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 8: Build a Neural Network for Manufacturing Sound</h1>
</div>

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1)

[Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

### ME 5995: AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science

Wayne State University

**Individual graded Lab**

**Version: Student Notebook**

[Open in Colab](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week09/lab08_neural_network_sound_classification.ipynb)

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Overview and objectives

Can a small neural network identify a machine type from its sound pattern?
Lab 7 learned a normal pattern for anomaly detection. Here all recordings are
normal and the target is machine type. The source dataset is familiar; the task differs.

By the end, you can explain the input/output shapes, apply training-only global
standardization, build a 32-neuron model, train it and interpret validation predictions.
Complete four checkpoints worth 10 points: CP1=2, CP2=2, CP3=4, CP4=2.
Provided cells run unchanged. Complete the marked code and short responses.
An unfinished executable `...` placeholder must be replaced before its cell can run.
This Lab ends with validation; the common test partition is reserved for Lab 9.

## Data and evaluation context

The supplied features represent **800 normal MIMII public 1.0 recordings**:
fan, pump, slider and valve; four machine IDs per type; 50 recordings per type/ID.
Each original recording lasts 10 seconds at 16 kHz, retaining source microphone
Channel 1. The source 0 dB condition contains mixed factory background noise.

Each parent recording gives ten nonoverlapping one-second segments. Each segment
is a 32-by-61 log-mel matrix: 32 mel bands and 61 time frames. The numerical log
power reference is digital, not calibrated dB SPL. Extraction is already complete.

Labels are **0=fan, 1=pump, 2=slider, 3=valve**. Machine-type classification is a
course-derived task, not the original MIMII anomalous-sound benchmark.
Only the log-mel values enter the NN. IDs, labels and partition names are metadata.

The fixed same-ID split has 480 training, 160 validation and 160 test recordings,
or 4,800/1,600/1,600 segments. All ten segments from one parent stay together.
The same ID string in different machine types does not identify one physical machine.
Previously used course recordings are not an untouched external benchmark.

**Software/files:** Colab provides TensorFlow/Keras, NumPy, pandas, Matplotlib
and scikit-learn. The supplied feature folder contains `logmel.npz`,
`recordings.csv`, `baseline32_summary.csv`, `FEATURE_DICTIONARY.md` and `LICENSE_DATA.txt`.
No package installation is part of the assessed work.

## Provided setup: run unchanged

The next two cells import packages and locate the supplied files.
This is technical infrastructure, not code you need to reconstruct.
The first learning section then opens the files and inspects what they contain.
No model is trained in this setup block.

In [ ]:
# PROVIDED: a small, reproducible CPU example and familiar plotting tools.
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from IPython.display import display
tf.get_logger().setLevel("ERROR")
tf.config.experimental.enable_op_determinism()

In [ ]:
# PROVIDED: use cached files, or retrieve the supplied course data.
data_base_url = "https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/mimii_logmel800/"
data_folder = Path("mimii_logmel800")
data_folder.mkdir(parents=True, exist_ok=True)
feature_file = data_folder / "logmel.npz"
metadata_file = data_folder / "recordings.csv"

In [ ]:
# PROVIDED: download only missing files. Do not edit this setup cell.
from urllib.request import urlretrieve
if not feature_file.is_file():
    urlretrieve(data_base_url + "logmel.npz", feature_file)
if not metadata_file.is_file():
    urlretrieve(data_base_url + "recordings.csv", metadata_file)
reference_file = data_folder / "baseline32_summary.csv"
if not reference_file.is_file():
    urlretrieve(data_base_url + "baseline32_summary.csv", reference_file)
dictionary_file = data_folder / "FEATURE_DICTIONARY.md"
if not dictionary_file.is_file():
    urlretrieve(data_base_url + "FEATURE_DICTIONARY.md", dictionary_file)
license_file = data_folder / "LICENSE_DATA.txt"
if not license_file.is_file():
    urlretrieve(data_base_url + "LICENSE_DATA.txt", license_file)

## First: open the files and understand the data

An `.npz` file is a container of named NumPy arrays. Opening it does not train or
standardize anything. `archive.files` shows the stored names; square brackets
retrieve one named array. The CSV is a separate recording-level metadata table.
Inspect these structures before applying a split or a transformation.

In [ ]:
# PROVIDED: open the feature container and the recording metadata.
archive = np.load(feature_file)
recordings = pd.read_csv(metadata_file)
print("Arrays stored in the NPZ file:", archive.files)
print("Metadata rows and columns:", recordings.shape)

### 1. Read the named arrays

`X` and `y` describe **segments**. `recording_id` and `machine_id` describe
**parent recordings**. `recording_index` links each segment to a parent.
Keep these names separate: an input row is not an independent recording.

In [ ]:
# PROVIDED: extract one named item at a time.
X = archive["X"]
y = archive["y"]
recording_index = archive["recording_index"]
recording_ids = archive["recording_id"]
recording_machine_ids = archive["machine_id"]
class_names = archive["class_names"]

In [ ]:
# PROVIDED: shape tells how many entries each array contains.
print("X shape and dtype:", X.shape, X.dtype)
print("y shape and dtype:", y.shape, y.dtype)
print("recording_index shape:", recording_index.shape)
print("recording_ids shape:", recording_ids.shape)
print("recording_machine_ids shape:", recording_machine_ids.shape)
print("class_names shape:", class_names.shape)

| Array | Shape | Meaning |
|---|---|---|
| X | (8000, 32, 61) | 8000 segments, 32 mel bands, 61 time frames |
| y | (8000,) | One integer machine-type label per segment |
| recording_index | (8000,) | One parent-recording row number per segment |
| recording_ids | (800,) | One stable identifier per parent recording |
| recording_machine_ids | (800,) | One machine ID per parent recording |
| class_names | (4,) | Meaning of label indices 0, 1, 2 and 3 |

`float32` stores numerical log-mel values. `int64` stores integer labels/indices.
A shape `(8000,)` is a one-dimensional array. In X, `X[0]` selects one 32-by-61
matrix; it does not select one frequency band from every segment.

In [ ]:
# PROVIDED: an integer label is an index into class_names.
label_key = pd.DataFrame()
label_key["Label"] = [0, 1, 2, 3]
label_key["Machine type"] = class_names
display(label_key)

### 2. Inspect the recording table

The table has 800 rows, one per 10-second parent recording. A recording ID
identifies a source file. A machine ID identifies a model/ID group within its
machine type. `partition` is the supplied train/validation/test assignment.
These columns support grouping and interpretation; they are not NN inputs.

In [ ]:
# PROVIDED: show a few useful columns, not all source checksums and file paths.
metadata_columns = ["recording_id", "machine_type", "machine_id", "label", "partition"]
metadata_preview = recordings[metadata_columns]
display(metadata_preview.head())

In [ ]:
# PROVIDED: count parent recordings before looking at segment counts.
recording_types = recordings["machine_type"]
metadata_machine_ids = recordings["machine_id"]
type_id_counts = pd.crosstab(recording_types, metadata_machine_ids)
display(type_id_counts)

### 3. Follow one segment back to its parent

The next example inspects the first stored row only to understand file structure.
It is not model evaluation. `recording_index[0]` is the parent-row number.
Using that number in `recording_ids` finds the source recording. No parsing of
filenames is needed. The corresponding class name comes from y, not the ID.

In [ ]:
# PROVIDED: inspect one segment and its parent using explicit variables.
example_segment = X[0]
example_label = y[0]
example_parent_index = recording_index[0]
example_parent_id = recording_ids[example_parent_index]
example_machine_id = recording_machine_ids[example_parent_index]
example_machine_type = class_names[example_label]
print("One segment shape:", example_segment.shape)
print("Label and machine type:", example_label, example_machine_type)
print("Parent row and recording ID:", example_parent_index, example_parent_id)
print("Parent machine ID:", example_machine_id)

In [ ]:
# PROVIDED: look at the same unstandardized segment as a matrix image.
plt.figure(figsize=(7, 3))
frame_centers_seconds = (np.arange(61) * 256 + 256) / 16000
mel_band_numbers = np.arange(32)
plt.pcolormesh(frame_centers_seconds, mel_band_numbers, example_segment, shading="nearest")
plt.xlabel("Time within segment (s)")
plt.ylabel("Mel band index")
plt.colorbar(label="Log-mel power (dB, numerical digital reference)")
plt.tight_layout()
plt.show()

### 4. Check why 800 recordings give 8000 segments

Rows 0 through 9 come from one parent recording. Row 10 begins the next parent.
The short preview below shows this relationship. Looking up an array with
several integer indices retrieves one value for each supplied index.
All ten segments from one parent must remain in the same partition.

In [ ]:
# PROVIDED: inspect only twelve rows, rather than printing the full dataset.
segment_rows = np.arange(12)
parent_indices = recording_index[:12]
parent_ids = recording_ids[parent_indices]
segment_labels = y[:12]
lineage_preview = pd.DataFrame()
lineage_preview["Segment row"] = segment_rows
lineage_preview["Parent row"] = parent_indices
lineage_preview["Recording ID"] = parent_ids
lineage_preview["Label"] = segment_labels
display(lineage_preview)

### Provided grouping: recording first

`recording_index` connects each segment to one parent row in `recordings`.
The following indexing copies the parent's partition to every segment. A mask
is a True/False array that selects rows. Do not create a random segment split.

In [ ]:
# PROVIDED: every segment receives its parent recording's partition.
recording_partitions = recordings["partition"].to_numpy()
segment_partitions = recording_partitions[recording_index]
train_mask = segment_partitions == "train"
validation_mask = segment_partitions == "validation"
test_mask = segment_partitions == "test"

In [ ]:
# PROVIDED: each True/False mask selects the matching segment rows.
X_train = X[train_mask]
y_train = y[train_mask]
X_validation = X[validation_mask]
y_validation = y[validation_mask]
X_test = X[test_mask]
y_test = y[test_mask]
print("Segment counts:", len(y_train), len(y_validation), len(y_test))

### One training-global transformation

Compute one mean and one population standard deviation across **all elements of
X_train**. Apply the same pair everywhere. This keeps segment-to-segment level
and contrast differences. It does not make every segment mean 0/SD 1, and it
does not force a Gaussian distribution. `mean()` and `std()` without an axis
combine every element of the supplied array.

In [ ]:
# GUIDED: a small complete example of one shared affine transformation.
example_values = np.array([2.0, 4.0, 6.0, 8.0])
example_mean = example_values.mean()
example_std = example_values.std()
example_scaled = (example_values - example_mean) / example_std
print("Example scaled values:", example_scaled)

## CP1: Prepare the inputs

Complete the global transformation. Reuse the same training statistics.

**API syntax hint:**

```python
values.mean()
values.std()
scaled_inputs = (inputs - fitted_mean) / fitted_std
```

In [ ]:
# Fit preprocessing using training values; reuse it on held-out inputs.
# TODO: use only X_train to compute the two scalar statistics.
training_mean = ...
training_std = ...
X_train_scaled = ...
X_validation_scaled = ...

In [ ]:
# PROVIDED: standardized training statistics should be close to 0 and 1.
print("Training mean and SD used:", training_mean, training_std)
print("Standardized training mean:", X_train_scaled.mean())
print("Standardized training SD:", X_train_scaled.std())
print("Shape after scaling:", X_train_scaled.shape)

**CP1 response:** State the segment count in each partition and explain which values may determine the global statistics.

**Your response:** TODO.

## CP2: Connect shapes to a model

Flatten converts each 32-by-61 matrix into 1,952 values without learning weights.
The batch dimension is the number of segments processed together, not another feature.
A Dense layer with p inputs and q outputs has p*q weights and q biases.
The hidden ReLU learns nonlinear combinations. Four softmax outputs are class
probabilities in `class_names` order; they are not fault-severity values.

In [ ]:
# GUIDED: an analogous small model with different dimensions and three classes.
example_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3, 4)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax")
])
example_model.summary()

Build one 32-unit ReLU hidden layer followed by four softmax outputs.

**API syntax hint:**

```python
tf.keras.Sequential([layer_1, layer_2, ...])
tf.keras.layers.Input(shape=(rows, columns))
tf.keras.layers.Flatten()
tf.keras.layers.Dense(units, activation="activation_name")
```

In [ ]:
# Define the layers for one input; Keras supplies the batch dimension.
# Clear framework state before constructing a fresh model.
tf.keras.backend.clear_session()
# Keep initialization reproducible for the controlled comparison.
tf.keras.utils.set_random_seed(9)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 61)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(..., activation=...),
    tf.keras.layers.Dense(..., activation=...)
])

In [ ]:
model.summary()

**CP2 response:** Calculate the parameter count of each Dense layer and explain why the output has four values.

**Your response:** TODO.

## CP3: Compile and train

`compile()` specifies how the model learns. Adam uses the fixed learning rate
0.001. Integer labels require `sparse_categorical_crossentropy`; the softmax
outputs are probabilities, so do not request a logits loss.
`fit()` updates weights from training data. Validation monitors predictions
without updating the weights. An epoch traverses the training set; batch size
32 specifies the number of segments per update. Use 40 epochs as a fixed example.

API pattern: `model.compile(optimizer=..., loss=..., metrics=["accuracy"])`.
Then `model.fit(inputs, labels, validation_data=(val_inputs, val_labels), ...)`.

Complete the loss name. Keep the optimizer and settings fixed.

**API syntax hint:**

```python
optimizer = tf.keras.optimizers.Adam(learning_rate=rate)
model.compile(optimizer=optimizer, loss="loss_name", metrics=["accuracy"])
```

In [ ]:
# Configure learning and reported metrics; compile does not update weights.
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer,
              loss=..., metrics=["accuracy"])

Complete training inputs/labels and the validation pair.

**API syntax hint:**

```python
history = model.fit(
    inputs, targets, validation_data=(validation_inputs, validation_targets),
    epochs=number_of_epochs, batch_size=batch_size, verbose=2
)
```

`verbose=2` prints one summary per epoch, including training and validation metrics. This changes the display, not the learning settings.

In [ ]:
# Update weights on training data; validation monitors held-out performance.
history = model.fit(
    ..., ..., validation_data=...,
    epochs=40, batch_size=32, shuffle=True, verbose=2
)

The Keras history stores named lists from training.
Read each list into a variable before plotting it. The plots below use epoch
numbers 1 through 40; Python list positions still start at zero.

In [ ]:
# PROVIDED: retrieve each numerical series separately.
training_losses = history.history["loss"]
validation_losses = history.history["val_loss"]
training_accuracies = history.history["accuracy"]
validation_accuracies = history.history["val_accuracy"]
epoch_numbers = np.arange(1, 41)

In [ ]:
# PROVIDED: an explicit loss plot, without a plotting helper or loop.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_losses, label="Training")
plt.plot(epoch_numbers, validation_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.title("Fixed 32-neuron example: loss")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# PROVIDED: accuracy is a different metric from loss.
plt.figure(figsize=(7, 3))
plt.plot(epoch_numbers, training_accuracies, label="Training")
plt.plot(epoch_numbers, validation_accuracies, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Segment accuracy")
plt.title("Fixed 32-neuron example: accuracy")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
final_training_loss = training_losses[-1]
final_validation_loss = validation_losses[-1]
print("Final training loss:", final_training_loss)
print("Final validation loss:", final_validation_loss)

**CP3 response:** Record the final training and validation loss. Describe what the curves show about training fit and held-out validation performance.

**Your response:** TODO.

## CP4: Predict and interpret

`model.predict(inputs, verbose=0)` returns one row of four probabilities per
segment. `argmax(axis=1)` picks the largest entry within each row and therefore
the predicted class index. This Lab evaluates validation only and uses final
epoch weights. Lab 9 will select and restore a checkpoint before test evaluation.

Predict probabilities for standardized validation inputs.

**API syntax hint:**

```python
probabilities = model.predict(inputs, verbose=0)
predicted_labels = np.argmax(probabilities, axis=class_axis)
```

In [ ]:
# Obtain probabilities before converting them to class labels.
validation_probabilities = model.predict(..., verbose=0)

In [ ]:
validation_predictions = np.argmax(validation_probabilities, axis=1)
validation_accuracy = accuracy_score(y_validation, validation_predictions)
print("Validation segment accuracy:", validation_accuracy)
validation_confusion = confusion_matrix(y_validation, validation_predictions, labels=[0, 1, 2, 3])
validation_display = ConfusionMatrixDisplay(validation_confusion, display_labels=class_names)
validation_display.plot()
plt.title("Validation segments: actual rows, predicted columns")
plt.tight_layout()
plt.show()

### Supplied classical reference

Logistic regression uses the same standardized flattened inputs, training labels
and validation partition. The fitting/scoring below is supplied for comparison;
do not tune it. A neural network does not have to beat this reference.

In [ ]:
# PROVIDED: prepare the same inputs for the tabular classical model.
number_of_training_segments = len(y_train)
number_of_validation_segments = len(y_validation)
X_train_flat = X_train_scaled.reshape(number_of_training_segments, 1952)
X_validation_flat = X_validation_scaled.reshape(number_of_validation_segments, 1952)

In [ ]:
from sklearn.linear_model import LogisticRegression
classical_model = LogisticRegression(max_iter=1000)
classical_model.fit(X_train_flat, y_train)
classical_predictions = classical_model.predict(X_validation_flat)
classical_accuracy = accuracy_score(y_validation, classical_predictions)
print("Classical validation segment accuracy:", classical_accuracy)
print("NN validation segment accuracy:", validation_accuracy)

**CP4 response:** Compare the two validation accuracies, name one actual off-diagonal confusion if present, and state why this is not an unseen-machine or fault-diagnosis result.

**Your response:** TODO.

## Optional challenge: inspect a probability vector

Inspect one correctly predicted validation/test segment and, if available, one
incorrectly predicted segment. Which two classes have the highest probabilities?
If all predictions are correct, compare two correct examples instead.
No extra model training, submission or points are required.

## Submission checklist

Rendered notebook checkboxes are not clickable. To record completion, edit this
Markdown cell and change `[ ]` to `[x]`.

- [ ] Enter your name and WSU AccessID.
- [ ] Complete required code and short written responses.
- [ ] Keep the required model/history/metric outputs visible.
- [ ] Restart the runtime and run cells from top to bottom after completing TODOs.
- [ ] Save `Lab08_Firstname_Lastname.ipynb` and its matching PDF.
- [ ] Submit both files through Canvas as instructed. Do not submit through GitHub.

No separate report is required. Optional work is ungraded. PDF rendering issues
receive corrective feedback; the notebook is used to assess obscured evidence.

## References and attribution

- [MIMII public 1.0](https://zenodo.org/records/3384388), DOI 10.5281/zenodo.3384388.
- Purohit et al., [MIMII dataset paper](https://arxiv.org/abs/1909.09347), 2019.
- [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).
- [Keras Sequential guide](https://keras.io/guides/sequential_model/).

Original authors: Harsh Purohit, Ryo Tanabe, Kenji Ichige, Takashi Endo,
Yuki Nikaido, Kaori Suefusa and Yohei Kawaguchi (Hitachi, Ltd.).
Course changes: selected normal 0-dB recordings, retained Channel 1, segmented
each recording into ten one-second inputs, computed log-mel features, and added
labels, metadata and a recording-level split. Preserve attribution and ShareAlike
terms when redistributing derived data. See supplied `FEATURE_DICTIONARY.md` and
`LICENSE_DATA.txt`. This is a teaching task, not an official MIMII benchmark.